# 01 · Time-fraction elicitation (Section 3.1.1, Appendix C.1)

Elicits weekly hours per task from **three models** (Claude Sonnet 4.6, GPT-5.4-mini, Gemini-3.1-flash-lite) with Tamkin & McCrory's disclosed prompt. Each occupation is queried with its **complete** O\*NET task list, and the replies are mapped back to the 212 register tasks. The three-model mean (`cross_model_mean_hours`) becomes `task_time_fraction` in the task master; it is renormalised within each scope only at the point of use (final_pipeline, Section 3.2).

**Reproducible without API keys.** Every model reply is cached in `inputs/time_fraction/elicitation_cache.json` (keyed `model||SOC||title`). The notebook only calls an API for a reply that is missing from the cache, so re-running it rebuilds the July 2026 results exactly. To re-elicit from scratch, point `CACHE_FILE` at a new file and set `ANTHROPIC_API_KEY`, `OPENAI_API_KEY` and `GEMINI_API_KEY` in your environment (never in the notebook).

| Output (`outputs/01_time_fraction/`) | Contents |
|---|---|
| `elicited_time_fractions.csv` | one row per register task: per-model hours, 3-model mean, SD, range, CV → **handed off** to `final_pipeline/data/` (Table D6) |
| `model_hours_long.csv` | every cached reply, one row per model × occupation × task (all O\*NET tasks of the 10 occupations) |
| `agreement_summary.csv` | Krippendorff's α (ordinal) and coverage counts |
| `agreement_report.txt` | the printed agreement report incl. the 15 highest-disagreement tasks |

Reads: `inputs/final_task_register.csv`, `inputs/onet/task_statements.xlsx`, the cache. Independent of notebooks 02–04.

In [1]:
import json
import logging
import re
import time
import os
from pathlib import Path

import pandas as pd
import krippendorff

import prep_common as pc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", force=True)
log = logging.getLogger("time_fraction")

## Configuration (Appendix C.1)

Prompt, models and settings exactly as used in the production run of 15 July 2026: one user message, no system prompt, provider-default temperature, `max_tokens=4096` for Claude only, one run per model per occupation.

In [2]:
CACHE_FILE = pc.TF_CACHE_FILE
INCLUDE_TASK_TYPES = None   # None = Core and Supplemental tasks; or e.g. ["Core"]
SLEEP_SECONDS = 1.0

MODEL_IDS = {
    "claude": "claude-sonnet-4-6",
    "gpt": "gpt-5.4-mini",
    "gemini": "gemini-3.1-flash-lite",
}

PROMPT_TEMPLATE = """You are estimating how much time workers in the occupation "{occupation_title}" spend on each of their job tasks. Below is the complete list of tasks for this occupation. For each task, estimate how many hours per week a typical worker spends on it.

Important: Don't worry about making the hours sum to exactly 40 or any specific total - we'll normalize the results afterward. Just give your best estimate for each task independently based on what seems realistic.

Tasks: {tasks}

Return ONLY a JSON object mapping each task_id to your estimated hours per week, with no additional text, explanations, or commentary. Format: {{"task_id_1": hours, "task_id_2": hours, ...}}"""

## Step 1 – Load the register and the O\*NET task lists

In [3]:
register = pc.load_register()
onet = pc.load_onet_tasks()
if INCLUDE_TASK_TYPES is not None and pc.ONET_TASKTYPE_COL in onet.columns:
    onet = onet[onet[pc.ONET_TASKTYPE_COL].isin(INCLUDE_TASK_TYPES)]


def build_occupation_task_lists(onet_df: pd.DataFrame) -> dict:
    occ_tasks = {}
    for (soc, title), group in onet_df.groupby([pc.ONET_SOC_COL, pc.ONET_TITLE_COL]):
        occ_tasks[(soc, title)] = dict(zip(group[pc.ONET_TASKID_COL], group[pc.ONET_TASK_COL]))
    return occ_tasks


def find_r2r_task_occupations(register_df: pd.DataFrame, onet_df: pd.DataFrame) -> pd.DataFrame:
    merged = register_df.merge(
        onet_df[[pc.ONET_TASKID_COL, pc.ONET_SOC_COL, pc.ONET_TITLE_COL]],
        left_on="task_id", right_on=pc.ONET_TASKID_COL, how="left",
    )
    unmatched = merged[merged[pc.ONET_SOC_COL].isna()]["task_id"].unique().tolist()
    if unmatched:
        log.warning(f"{len(unmatched)} R2R task_id(s) not found in task_statements.xlsx: {unmatched}")
    return merged.dropna(subset=[pc.ONET_SOC_COL])


occ_task_lists = build_occupation_task_lists(onet)
r2r_occ_map = find_r2r_task_occupations(register, onet)
needed_occupations = sorted(set(zip(r2r_occ_map[pc.ONET_SOC_COL], r2r_occ_map[pc.ONET_TITLE_COL])))
print(f"{len(register)} register tasks in {len(needed_occupations)} occupations")
for soc, title in needed_occupations:
    print(f"  {soc}  {title:<45} {len(occ_task_lists[(soc, title)]):>3} O*NET tasks")

212 register tasks in 10 occupations
  11-1011.00  Chief Executives                               31 O*NET tasks
  11-3031.00  Financial Managers                             17 O*NET tasks
  11-3031.01  Treasurers and Controllers                     22 O*NET tasks
  13-1051.00  Cost Estimators                                14 O*NET tasks
  13-2011.00  Accountants and Auditors                       29 O*NET tasks
  13-2031.00  Budget Analysts                                13 O*NET tasks
  13-2082.00  Tax Preparers                                  12 O*NET tasks
  15-1242.00  Database Administrators                        18 O*NET tasks
  43-3021.00  Billing and Posting Clerks                     28 O*NET tasks
  43-3031.00  Bookkeeping, Accounting, and Auditing Clerks   28 O*NET tasks


## Step 2 – Prompt building, JSON parsing and model back-ends

Clients are created lazily, so the SDKs (`anthropic`, `openai`, `google-genai`) and keys are only needed on a cache miss.

In [4]:
def build_prompt(occupation_title: str, task_dict: dict) -> str:
    tasks_str = "\n".join(f"{tid}: {stmt}" for tid, stmt in task_dict.items())
    return PROMPT_TEMPLATE.format(occupation_title=occupation_title, tasks=tasks_str)


def parse_json_response(raw_text: str) -> dict:
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Could not parse JSON from model response:\n{raw_text[:500]}")


_clients = {}


def _client(name):
    if name not in _clients:
        if name == "anthropic":
            import anthropic
            _clients[name] = anthropic.Anthropic()            # reads ANTHROPIC_API_KEY
        elif name == "openai":
            import openai
            _clients[name] = openai.OpenAI()                  # reads OPENAI_API_KEY
        elif name == "gemini":
            from google import genai
            _clients[name] = genai.Client()                   # reads GEMINI_API_KEY
    return _clients[name]


def call_claude(prompt: str) -> str:
    response = _client("anthropic").messages.create(
        model=MODEL_IDS["claude"], max_tokens=4096,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


def call_gpt(prompt: str) -> str:
    response = _client("openai").chat.completions.create(
        model=MODEL_IDS["gpt"], messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


def call_gemini(prompt: str) -> str:
    response = _client("gemini").models.generate_content(model=MODEL_IDS["gemini"], contents=prompt)
    return response.text


# Registry -- everything downstream loops over this dict.
MODELS = {"claude": call_claude, "gpt": call_gpt, "gemini": call_gemini}


def elicit_hours(occupation_title: str, task_dict: dict, call_fn, model_key: str, max_retries: int = 3) -> dict:
    prompt = build_prompt(occupation_title, task_dict)
    for attempt in range(1, max_retries + 1):
        try:
            parsed = parse_json_response(call_fn(prompt))
            missing_keys = set(task_dict.keys()) - set(map(str, parsed.keys()))
            if missing_keys:
                log.warning(f"  [{model_key}/{occupation_title}] omitted {len(missing_keys)} task_id(s): {missing_keys}")
            return parsed
        except Exception as e:
            log.warning(f"  [{model_key}/{occupation_title}] attempt {attempt}/{max_retries} failed: {e}")
            time.sleep(2 * attempt)
    raise RuntimeError(f"Failed to elicit hours from {model_key} for '{occupation_title}' after {max_retries} attempts.")

## Step 3 – Elicitation (cache first, API only on a miss)

In [5]:
def load_cache(path: Path) -> dict:
    return json.loads(Path(path).read_text()) if Path(path).exists() else {}


def save_cache(cache: dict, path: Path):
    Path(path).write_text(json.dumps(cache, indent=2))


cache = load_cache(CACHE_FILE)
all_rows, n_api_calls = [], 0
for i, (soc, title) in enumerate(needed_occupations, 1):
    task_dict = occ_task_lists.get((soc, title), {})
    for model_key, call_fn in MODELS.items():
        cache_key = f"{model_key}||{soc}||{title}"
        if cache_key in cache:
            hours_map = cache[cache_key]
        else:
            log.info(f"[{i}/{len(needed_occupations)}] {title} ({soc}) -- calling {model_key}")
            hours_map = elicit_hours(title, task_dict, call_fn, model_key)
            cache[cache_key] = hours_map
            save_cache(cache, CACHE_FILE)
            n_api_calls += 1
            time.sleep(SLEEP_SECONDS)
        for task_id, hours in hours_map.items():
            all_rows.append({"onet_soc_code": soc, "occupation_title": title, "task_id": str(task_id),
                             "model": model_key, "hours": hours})

raw_df = pd.DataFrame(all_rows)
print(f"{len(raw_df)} replies ({raw_df['task_id'].nunique()} tasks x {raw_df['model'].nunique()} models); "
      f"{n_api_calls} new API calls, the rest from {CACHE_FILE.name}")
pc.save_csv(raw_df, "time_fraction", "model_hours_long.csv")

636 replies (212 tasks x 3 models); 0 new API calls, the rest from elicitation_cache.json
[write] outputs/01_time_fraction/model_hours_long.csv  (636 rows)


## Step 4 – Wide table, ensemble mean and per-task disagreement

`cross_model_mean_hours` is the unweighted mean of the three raw weekly-hour estimates; it is **not** renormalised here.

In [6]:
r2r_only = raw_df.merge(register[["task_id", "task_statement"]], on="task_id", how="inner")
wide = r2r_only.pivot_table(index=["task_id", "task_statement", "onet_soc_code", "occupation_title"],
                            columns="model", values="hours")
wide.columns = [f"{c}_hours" for c in wide.columns]
summary = wide.reset_index()

model_cols = [f"{m}_hours" for m in MODELS.keys()]
summary["cross_model_mean_hours"] = summary[model_cols].mean(axis=1)   # -> task_time_fraction
summary["cross_model_std_hours"] = summary[model_cols].std(axis=1)
summary["cross_model_range_hours"] = summary[model_cols].max(axis=1) - summary[model_cols].min(axis=1)
summary["cross_model_cv"] = summary["cross_model_std_hours"] / summary["cross_model_mean_hours"]

uncovered = set(register["task_id"]) - set(summary["task_id"])
if uncovered:
    log.warning(f"{len(uncovered)} register task_id(s) have NO elicited hours: {uncovered}")
pc.save_csv(summary, "time_fraction", "elicited_time_fractions.csv")
summary.head()

[write] outputs/01_time_fraction/elicited_time_fractions.csv  (212 rows)


,task_id,task_statement,onet_soc_code,occupation_title,claude_hours,gemini_hours,gpt_hours,cross_model_mean_hours,cross_model_std_hours,cross_model_range_hours,cross_model_cv
0,1298,Develop standards and guidelines for the use a...,15-1242.00,Database Administrators,2.0,2.5,2.0,2.166667,0.288675,0.5,0.133235
1,1299,Modify existing databases and database managem...,15-1242.00,Database Administrators,5.5,6.0,4.5,5.333333,0.763763,1.5,0.143205
2,1300,"Test programs or databases, correct errors, an...",15-1242.00,Database Administrators,4.5,6.0,2.5,4.333333,1.755942,3.5,0.405217
3,1301,"Plan, coordinate, and implement security measu...",15-1242.00,Database Administrators,4.0,5.5,3.5,4.333333,1.040833,2.0,0.240192
4,1302,"Approve, schedule, plan, and supervise the ins...",15-1242.00,Database Administrators,3.0,3.5,2.0,2.833333,0.763763,1.5,0.269563


## Step 5 – Cross-model agreement (Krippendorff's α, ordinal; Table D6)

One omnibus statistic across the three models on all 212 rows. Ordinal level, because the interest is whether the models rank tasks the same way rather than agree on exact hours. The 91-task in-scope value is reported alongside.

In [7]:
def kripp_ordinal(df):
    return krippendorff.alpha(reliability_data=df[model_cols].to_numpy(dtype=float).T,
                              level_of_measurement="ordinal")


in_ids = pc.in_scope_ids(register)
n_complete = summary[model_cols].dropna().shape[0]
alpha_all = kripp_ordinal(summary)
alpha_in = kripp_ordinal(summary[summary["task_id"].isin(in_ids)])
agreement = pd.DataFrame([
    {"scope": "all register tasks", "n_rows": len(summary), "n_rows_all_models": n_complete,
     "krippendorff_alpha_ordinal": alpha_all},
    {"scope": "in-scope tasks", "n_rows": int(summary["task_id"].isin(in_ids).sum()),
     "n_rows_all_models": int(summary[summary["task_id"].isin(in_ids)][model_cols].dropna().shape[0]),
     "krippendorff_alpha_ordinal": alpha_in},
])
pc.save_csv(agreement, "time_fraction", "agreement_summary.csv")

lines = ["=" * 72, "CROSS-MODEL AGREEMENT REPORT", "=" * 72, "",
         f"Task-occupation rows: {len(summary)} total, {n_complete} with all {len(model_cols)} models present.\n",
         f"Krippendorff's alpha (ordinal level, 3 raters, all rows incl. partial missingness): {alpha_all:.3f}",
         f"Krippendorff's alpha (ordinal, in-scope tasks only, n={agreement.loc[1, 'n_rows']}): {alpha_in:.3f}",
         "  -> Single omnibus agreement statistic across all 3 models at once.",
         "  -> Ordinal level of measurement: emphasizes whether models rank tasks the",
         "     same way (directional agreement), rather than requiring exact-value match.",
         "\nTop 15 highest-disagreement tasks (by coefficient of variation across models):"]
top_disagree = summary.sort_values("cross_model_cv", ascending=False).head(15)
lines.append(top_disagree[["task_id", "occupation_title", "task_statement"] + model_cols + ["cross_model_cv"]]
             .round(2).to_string(index=False))
report = "\n".join(lines)
pc.save_text(report, "time_fraction", "agreement_report.txt")
print(report)

[write] outputs/01_time_fraction/agreement_summary.csv  (2 rows)
[write] outputs/01_time_fraction/agreement_report.txt
CROSS-MODEL AGREEMENT REPORT

Task-occupation rows: 212 total, 212 with all 3 models present.

Krippendorff's alpha (ordinal level, 3 raters, all rows incl. partial missingness): 0.681
Krippendorff's alpha (ordinal, in-scope tasks only, n=91): 0.689
  -> Single omnibus agreement statistic across all 3 models at once.
  -> Ordinal level of measurement: emphasizes whether models rank tasks the
     same way (directional agreement), rather than requiring exact-value match.

Top 15 highest-disagreement tasks (by coefficient of variation across models):
task_id                             occupation_title                                                                                                                                                                                    task_statement  claude_hours  gpt_hours  gemini_hours  cross_model_cv
  21525                 

## Hand-off to the analysis phase

`final_pipeline/05_appendix_time_fraction_agreement.ipynb` builds Table D6 from this file.

In [8]:
pc.handoff("time_fraction", "elicited_time_fractions.csv")

[handoff] outputs/01_time_fraction/elicited_time_fractions.csv -> ../final_pipeline/data/elicited_time_fractions.csv
